In [ ]:
from pathlib import Path
import json
import urllib.request
import numpy as np
import pandas as pd
from IPython.display import display
import subprocess
import plotly.graph_objects as go

# Parameters to tune
START_DATE = "2026-06-30"
END_DATE = "2026-07-28"

UNCORRELATED_BPS_BY_CHAIN = {
    "ethereum": 4.0, "gnosis": 3.0, "arbitrum": 1.0, "base": 2.0,
    "avalanche_c": 2.0, "polygon": 3.0, "bnb": 1.0,
}
CORRELATED_BPS = 0.1

# Constants
TOKEN_LISTS_URL = "https://cms.cow.finance/api/correlated-tokens?pagination[pageSize]=100"
WEI = 1e18
DATA_DIR = Path("../data")
TOKEN_LISTS_PATH = DATA_DIR / "token_lists.json"
CHAIN_ALIASES = {
    "ethereum": ("mainnet", "ethereum"), "gnosis": ("gnosis", "xdai"),
    "arbitrum": ("arbitrum",), "base": ("base",), "polygon": ("polygon",),
    "bnb": ("bnb", "bsc"), "avalanche_c": ("avalanche",),
}
FETCH_SCRIPT = Path("../scripts/fetch_data.py")

def input_paths(data_dir, chain, start, end):
    """The three CSVs scripts/fetch_data.py writes per chain and window."""
    data_dir = Path(data_dir)
    return {
        "rewards": data_dir / f"{chain}_{start}_{end}.csv",
        "failed volumes": data_dir / f"{chain}_{start}_{end}_failed_volumes.csv",
        "consistency shares": data_dir / f"{chain}_{start}_{end}_consistency_shares.csv",
    }


def fetch_inputs_if_missing(
    start_date,
    end_date,
    chains,
    data_dir=DATA_DIR,
    fetch_script=FETCH_SCRIPT,
):
    data_dir = Path(data_dir)
    fetch_script = Path(fetch_script)
    start = pd.Timestamp(start_date).strftime("%Y-%m-%d")
    end = pd.Timestamp(end_date).strftime("%Y-%m-%d")
    data_dir.mkdir(parents=True, exist_ok=True)
    if not fetch_script.exists():
        raise FileNotFoundError(f"Fetch script not found: {fetch_script}")
    for chain in chains:
        paths = input_paths(data_dir, chain, start, end)
        if all(path.exists() for path in paths.values()):
            print(f"{chain}: using cached CSVs for {start} to {end}")
            continue
        print(f"{chain}: fetching data for {start} to {end}")
        subprocess.run(
            ["uv","run","python",str(fetch_script),"--chain",chain,"--start",start,"--end",end,"--out",str(paths["rewards"])],
            check=True,
        )
        for label, path in paths.items():
            if not path.exists():
                raise FileNotFoundError(
                    f"Fetch completed but the {label} CSV was not created: {path}"
                )


def load_inputs(
    data_dir,
    start_date,
    end_date,
    chains,
):
    data_dir = Path(data_dir)
    start = pd.Timestamp(start_date).strftime("%Y-%m-%d")
    end = pd.Timestamp(end_date).strftime("%Y-%m-%d")

    paths_by_chain = {
        chain: input_paths(data_dir, chain, start, end) for chain in chains
    }
    missing = [
        path
        for paths in paths_by_chain.values()
        for path in paths.values()
        if not path.exists()
    ]
    if missing:
        raise FileNotFoundError(
            "Missing input CSVs:\n" + "\n".join(str(path) for path in missing)
        )

    def read(label):
        for chain, paths in paths_by_chain.items():
            print(" ", paths[label].name)
        return pd.concat(
            [
                pd.read_csv(paths[label], low_memory=False)
                for paths in paths_by_chain.values()
            ],
            ignore_index=True,
        ).drop_duplicates()

    print("Reward files:")
    rewards = read("rewards")
    print("\nFailed-volume files:")
    volumes = read("failed volumes")
    print("\nConsistency-share files:")
    shares = read("consistency shares")

    return rewards, volumes, shares


def load_correlated_groups(chains):
    if not TOKEN_LISTS_PATH.exists():
        TOKEN_LISTS_PATH.parent.mkdir(parents=True, exist_ok=True)
        with urllib.request.urlopen(TOKEN_LISTS_URL) as response:
            TOKEN_LISTS_PATH.write_bytes(response.read())

    payload = json.loads(TOKEN_LISTS_PATH.read_text())

    named_groups = [
        (
            entry["attributes"]["name"].lower(),
            {str(token).lower() for token in entry["attributes"]["tokens"]},
        )
        for entry in payload["data"]
    ]

    groups_by_chain = {}
    for chain in chains:
        aliases = CHAIN_ALIASES[chain]
        groups = [
            tokens for name, tokens in named_groups
            if any(alias in name for alias in aliases)
        ]
        if not groups:
            raise ValueError(f"No correlated-token groups found for {chain}")
        groups_by_chain[chain] = groups

    return groups_by_chain


def build_auction_solver(rewards, volumes, groups_by_chain):
    """Collapse the two fetched datasets to one row per (chain, auction, solver).

    The rewards CSV is already at that grain; the failed-volume CSV is one row
    per token pair, which is what the correlated/uncorrelated split needs.
    """
    required_rewards = {
        "blockchain", "auction_id", "solver", "accounting_period",
        "is_excluded_from_penalties", "reward_penalty_native",
        "reward_penalty_uncapped_native", "reward_cap_upper_native",
    }
    missing = sorted(required_rewards - set(rewards.columns))
    if missing:
        raise KeyError(f"Reward CSVs are missing required columns: {missing}")

    required_volumes = {
        "blockchain", "auction_id", "solver",
        "sell_token", "buy_token", "failed_volume_native",
    }
    missing = sorted(required_volumes - set(volumes.columns))
    if missing:
        raise KeyError(f"Failed-volume CSVs are missing required columns: {missing}")

    data = rewards.copy()
    data["solver"] = data["solver"].astype(str).str.lower().str.strip()
    data["accounting_period"] = data["accounting_period"].astype(str)

    native_columns = [
        "reward_penalty_native", "reward_penalty_uncapped_native",
        "reward_cap_upper_native",
    ]
    for column in native_columns:
        data[column] = pd.to_numeric(data[column], errors="coerce") / WEI

    data["excluded"] = (
        data["is_excluded_from_penalties"].astype(str).str.lower()
        .map({"true": True, "false": False}).fillna(False)
    )

    failed = volumes.copy()
    failed["solver"] = failed["solver"].astype(str).str.lower().str.strip()
    failed["sell_token"] = failed["sell_token"].astype(str).str.lower().str.strip()
    failed["buy_token"] = failed["buy_token"].astype(str).str.lower().str.strip()
    failed["failed_volume_native"] = (
        pd.to_numeric(failed["failed_volume_native"], errors="coerce").fillna(0) / WEI
    )

    failed["correlated"] = False
    for chain, token_groups in groups_by_chain.items():
        chain_mask = failed["blockchain"].eq(chain)
        sell_tokens = failed.loc[chain_mask, "sell_token"]
        buy_tokens = failed.loc[chain_mask, "buy_token"]
        correlated = np.zeros(chain_mask.sum(), dtype=bool)
        for token_group in token_groups:
            correlated |= (
                sell_tokens.isin(token_group).to_numpy()
                & buy_tokens.isin(token_group).to_numpy()
            )
        failed.loc[chain_mask, "correlated"] = correlated

    failed["failed_correlated_native"] = np.where(
        failed["correlated"], failed["failed_volume_native"], 0.0,
    )
    failed["failed_uncorrelated_native"] = np.where(
        ~failed["correlated"], failed["failed_volume_native"], 0.0,
    )

    keys = ["blockchain", "auction_id", "solver"]
    volume_totals = failed.groupby(keys, as_index=False).agg(
        failed_correlated_native=("failed_correlated_native", "sum"),
        failed_uncorrelated_native=("failed_uncorrelated_native", "sum"),
    )

    batch = data.rename(
        columns={
            "reward_penalty_native": "current_batch_native",
            "reward_penalty_uncapped_native": "uncapped_batch_native",
            "reward_cap_upper_native": "upper_reward_cap_native",
        }
    )[
        keys + ["accounting_period", "excluded", "current_batch_native",
                "uncapped_batch_native", "upper_reward_cap_native"]
    ]

    result = batch.merge(volume_totals, on=keys, how="left", validate="one_to_one")
    volume_columns = ["failed_correlated_native", "failed_uncorrelated_native"]
    result[volume_columns] = result[volume_columns].fillna(0.0)
    return result


def build_scenarios(auction_solver):
    data = auction_solver.copy()

    configured_chains = set(UNCORRELATED_BPS_BY_CHAIN)
    loaded_chains = set(data["blockchain"].dropna().unique())
    missing_rates = sorted(loaded_chains - configured_chains)
    if missing_rates:
        raise ValueError(f"No uncorrelated-token rate configured for chains: {missing_rates}")

    data["performance_reward_native"] = data["current_batch_native"].clip(lower=0)
    data["uncapped_penalty_native"] = (-data["uncapped_batch_native"]).clip(lower=0)

    current = data.assign(
        scenario="current", uncorrelated_rate_bps=np.nan,
        correlated_rate_bps=np.nan, penalty_cap_native=np.nan,
        penalty_native=(-data["current_batch_native"]).clip(lower=0),
        net_batch_native=data["current_batch_native"],
    )

    uncorrelated_rate = data["blockchain"].map(UNCORRELATED_BPS_BY_CHAIN).astype(float)
    proposed_cap = (
        uncorrelated_rate / 1e4 * data["failed_uncorrelated_native"]
        + CORRELATED_BPS / 1e4 * data["failed_correlated_native"]
    )
    proposed_penalty = np.minimum(data["uncapped_penalty_native"], proposed_cap)
    proposed_penalty = np.where(data["excluded"], 0.0, proposed_penalty)

    proposed = data.assign(
        scenario="proposed", uncorrelated_rate_bps=uncorrelated_rate,
        correlated_rate_bps=CORRELATED_BPS, penalty_cap_native=proposed_cap,
        penalty_native=proposed_penalty,
        net_batch_native=data["performance_reward_native"] - proposed_penalty,
    )

    result = pd.concat([current, proposed], ignore_index=True)
    result["consistency_budget_native"] = (
        result["upper_reward_cap_native"] - result["net_batch_native"]
    )

    negative_budget = result[result["consistency_budget_native"] < -1e-9]
    if not negative_budget.empty:
        raise ValueError(
            "Negative consistency budget found:\n"
            + negative_budget[
                ["blockchain", "auction_id", "solver", "scenario",
                 "consistency_budget_native"]
            ].head(20).to_string(index=False)
        )

    return result


def prepare_consistency_shares(shares):
    shares = shares.copy()
    shares["solver"] = shares["solver"].astype(str).str.lower().str.strip()

    direct_share = pd.to_numeric(shares["consistency_reward_share"], errors="coerce")
    total_budget = pd.to_numeric(shares["total_consistency_budget"], errors="coerce")
    solver_reward = pd.to_numeric(shares["consistency_reward_native"], errors="coerce")
    calculated_share = solver_reward / total_budget.replace(0, np.nan)

    shares["consistency_reward_share"] = (
        direct_share.fillna(calculated_share).fillna(0)
    )

    return shares.groupby(
        ["blockchain", "accounting_period", "solver"], as_index=False,
    ).agg(consistency_reward_share=("consistency_reward_share", "first"))


def calculate_solver_payments(scenarios, shares):
    required = {"blockchain", "accounting_period", "solver", "consistency_reward_share"}
    missing = sorted(required - set(shares.columns))
    if missing:
        raise KeyError(f"Consistency CSVs are missing columns: {missing}")

    share_data = prepare_consistency_shares(shares)

    weekly_budget = scenarios.groupby(
        ["blockchain", "accounting_period", "scenario"], as_index=False,
    ).agg(consistency_budget_native=("consistency_budget_native", "sum"))

    share_sums = share_data.groupby(
        ["blockchain", "accounting_period"], as_index=False,
    ).agg(share_sum=("consistency_reward_share", "sum"))

    share_check = weekly_budget.merge(
        share_sums, on=["blockchain", "accounting_period"], how="left",
    )
    bad_periods = share_check[
        share_check["consistency_budget_native"].abs().gt(1e-9)
        & ~np.isclose(share_check["share_sum"].fillna(0), 1.0, atol=1e-8)
    ]
    if not bad_periods.empty:
        raise ValueError(
            "Consistency shares do not sum to one:\n"
            + bad_periods[
                ["blockchain", "accounting_period", "share_sum"]
            ].drop_duplicates().to_string(index=False)
        )

    allocated = share_data.merge(
        weekly_budget, on=["blockchain", "accounting_period"], how="inner",
    )
    allocated["consistency_reward_native"] = (
        allocated["consistency_reward_share"] * allocated["consistency_budget_native"]
    )

    weekly_batch = scenarios.groupby(
        ["blockchain", "accounting_period", "solver", "scenario"], as_index=False,
    ).agg(
        performance_reward_native=("performance_reward_native", "sum"),
        penalty_native=("penalty_native", "sum"),
        net_batch_native=("net_batch_native", "sum"),
    )

    payments = weekly_batch.merge(
        allocated[
            ["blockchain", "accounting_period", "solver", "scenario",
             "consistency_reward_native"]
        ],
        on=["blockchain", "accounting_period", "solver", "scenario"],
        how="outer",
    )

    numeric_columns = [
        "performance_reward_native", "penalty_native",
        "net_batch_native", "consistency_reward_native",
    ]
    payments[numeric_columns] = payments[numeric_columns].fillna(0)
    payments["total_payment_native"] = (
        payments["net_batch_native"] + payments["consistency_reward_native"]
    )

    result = payments.groupby(
        ["blockchain", "solver", "scenario"], as_index=False,
    ).agg(
        performance_reward_native=("performance_reward_native", "sum"),
        penalty_native=("penalty_native", "sum"),
        consistency_reward_native=("consistency_reward_native", "sum"),
        total_payment_native=("total_payment_native", "sum"),
    )

    current_payment = (
        result[result["scenario"].eq("current")]
        [["blockchain", "solver", "total_payment_native"]]
        .rename(columns={"total_payment_native": "current_total_native"})
    )

    result = result.merge(current_payment, on=["blockchain", "solver"], how="left")
    result["change_vs_current_native"] = (
        result["total_payment_native"] - result["current_total_native"].fillna(0)
    )

    totals = result.groupby(["blockchain", "scenario"])["total_payment_native"].sum().unstack()
    max_difference = totals.sub(totals["current"], axis=0).abs().max().max()
    assert max_difference < 1e-8, totals
    print("Total payments are unchanged across scenarios.")

    return result


def run_counterfactual(start_date=START_DATE, end_date=END_DATE, data_dir=DATA_DIR, chains=None,fetch_script=FETCH_SCRIPT, make_plots=True):
    if chains is None:
        chains = list(CHAIN_ALIASES)
    fetch_inputs_if_missing(start_date=start_date,end_date=end_date,chains=chains,data_dir=data_dir,fetch_script=fetch_script)
    rewards, volumes, shares = load_inputs(data_dir=data_dir,start_date=start_date,end_date=end_date,chains=chains)

    chains = sorted(rewards["blockchain"].dropna().unique())
    unknown_chains = sorted(set(chains) - set(CHAIN_ALIASES))
    if unknown_chains:
        raise ValueError(f"Missing CHAIN_ALIASES entries for: {unknown_chains}")

    correlated_groups = load_correlated_groups(chains)
    auction_solver = build_auction_solver(rewards, volumes, correlated_groups)
    auction_scenarios = build_scenarios(auction_solver)
    solver_payments = calculate_solver_payments(auction_scenarios, shares)

    scenario_order = pd.CategoricalDtype(["current", "proposed"], ordered=True)
    solver_payments["scenario"] = solver_payments["scenario"].astype(scenario_order)

    for chain in chains:
        table = solver_payments[solver_payments["blockchain"].eq(chain)].copy()

        current_ranking = (
            table[table["scenario"].eq("current")]
            [["solver", "total_payment_native"]]
            .rename(columns={"total_payment_native": "_rank"})
        )

        table = (
            table.merge(current_ranking, on="solver", how="left")
            .sort_values(
                ["_rank", "solver", "scenario"],
                ascending=[False, True, True],
            )
            .drop(columns="_rank")
        )

        rate = UNCORRELATED_BPS_BY_CHAIN[chain]
        print(
            f"\n=== {chain} | uncorrelated={rate:g} bps, "
            f"correlated={CORRELATED_BPS:g} bps ==="
        )

        display(
            table[
                ["solver", "scenario", "performance_reward_native",
                 "penalty_native", "consistency_reward_native",
                 "total_payment_native", "change_vs_current_native"]
            ].round(6)
        )
        if make_plots:
            plot_solver_counterfactual(
                solver_payments,
                start_date,
                end_date,
            )

    return solver_payments, auction_scenarios

def plot_solver_counterfactual(
    solver_payments,
    start_date,
    end_date,
):
    start = pd.Timestamp(start_date).strftime("%Y-%m-%d")
    end = pd.Timestamp(end_date).strftime("%Y-%m-%d")
    for chain in sorted(solver_payments["blockchain"].dropna().unique()):
        chain_data = solver_payments[
            solver_payments["blockchain"].eq(chain)
        ].copy()
        chart_data = (
            chain_data.pivot_table(
                index="solver",
                columns="scenario",
                values="total_payment_native",
                aggfunc="sum",
                fill_value=0,
            )
            .reset_index()
        )

        for scenario in ["current", "proposed"]:
            if scenario not in chart_data.columns:
                chart_data[scenario] = 0.0

        chart_data = chart_data.sort_values("current",ascending=False)
        width = max(1100, 30 * len(chart_data) + 300)
        fig = go.Figure()
        fig.add_bar(x=chart_data["solver"],y=chart_data["current"],name="Current")
        fig.add_bar(x=chart_data["solver"],y=chart_data["proposed"],name="Proposed")
        fig.add_hline(y=0,line_width=0.8)
        fig.update_xaxes(title_text="Solver address",tickangle=-90)
        fig.update_layout(
            title=(
                "Solvers PnL Counterfactual — "
                "Proposed Penalty Caps — "
                f"{chain} — {start} to {end}"
            ),
            yaxis_title="Total payment (native token)",
            barmode="group",
            template="plotly_white",
            width=width,
            height=700,
            legend_title_text="Scenario",
        )
        fig.show()

SOLVER_PAYMENTS, AUCTION_SCENARIOS = run_counterfactual()